# Data Vortex A'26 - Round 2: NLP Sentiment Classification
**Team: Event Horizon**

This notebook contains the complete code used for developing and training both the **Local Machine Learning Model** (TF-IDF + Logistic Regression) and the **Deep Learning Benchmark Model** (BERTweet).

The pipeline is split into the following stages:
1. Data Preprocessing & Cleaning
2. Local Model: Training and Cross-Validation (TF-IDF + Logistic Regression)
3. Local Model: Evaluation and Error Analysis
4. Benchmark Model: BERTweet (vinai/bertweet-base)


## 1. Data Preprocessing
The following cell contains the code for cleaning text (removing HTML, mentions, URLs, emojis, etc.), plotting EDA, and splitting the dataset into 80/20 train/test splits.


In [ ]:
"""
preprocess.py — Round 2 NLP Pipeline
Data Vortex A'26 | Team: Event Horizon

Handles: text cleaning, EDA visualizations, stratified train/test split.
No leakage: all transformations fitted on train set only.
"""

import re
import os
import time
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "Data", "Labeled_Social_NLP_Training_Data.csv")
FIGURES_DIR = os.path.join(os.path.dirname(__file__), "..", "reports", "figures")
PROCESSED_DIR = os.path.join(os.path.dirname(__file__), "..", "reports")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ──────────────────────────────────────────────
# Text Cleaning
# ──────────────────────────────────────────────

_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_HTML_RE = re.compile(r"<[^>]+>|&amp;|&lt;|&gt;|&nbsp;|&quot;")
_MENTION_RE = re.compile(r"@\w+")
_HASHTAG_RE = re.compile(r"#(\w+)")
_EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
    "\u2600-\u26FF\u2700-\u27BF]+",
    flags=re.UNICODE,
)
_PUNCT_RE = re.compile(r'[^\w\s]')
_MULTI_SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = _HTML_RE.sub(" ", text)
    text = _URL_RE.sub(" ", text)
    text = _MENTION_RE.sub(" ", text)
    text = _HASHTAG_RE.sub(r" \1 ", text)
    text = _EMOJI_RE.sub(" ", text)
    text = text.lower()
    text = _PUNCT_RE.sub(" ", text)
    text = _MULTI_SPACE_RE.sub(" ", text).strip()
    return text


# ──────────────────────────────────────────────
# Load & Audit Dataset
# ──────────────────────────────────────────────

def load_and_audit(path: str) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(path, encoding="utf-8")
    audit = {}

    audit["raw_rows"] = len(df)
    audit["columns"] = list(df.columns)
    audit["label_counts"] = df["sentiment_label"].value_counts().to_dict()
    audit["topic_counts"] = df["topic_category"].value_counts().to_dict()

    missing_text = df["post_text"].isna().sum()
    audit["missing_text_rows"] = int(missing_text)
    df = df.dropna(subset=["post_text"])

    empty_after_strip = (df["post_text"].str.strip() == "").sum()
    audit["empty_text_rows"] = int(empty_after_strip)
    df = df[df["post_text"].str.strip() != ""]

    before_dedup = len(df)
    df = df.drop_duplicates(subset=["post_text"])
    audit["duplicate_rows_removed"] = before_dedup - len(df)

    audit["clean_rows"] = len(df)
    return df, audit


# ──────────────────────────────────────────────
# Apply Cleaning
# ──────────────────────────────────────────────

def apply_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["cleaned_text"] = df["post_text"].apply(clean_text)
    df["word_count"] = df["cleaned_text"].apply(lambda x: len(x.split()))
    df["char_count"] = df["cleaned_text"].apply(len)
    empty_after_clean = (df["cleaned_text"].str.strip() == "").sum()
    df = df[df["cleaned_text"].str.strip() != ""]
    return df, empty_after_clean


# ──────────────────────────────────────────────
# EDA Visualizations
# ──────────────────────────────────────────────

PALETTE = {"Positive": "#2ecc71", "Negative": "#e74c3c", "Neutral": "#3498db"}
TOPIC_PALETTE = sns.color_palette("husl", 8)


def plot_class_distribution(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0f1117")
    for ax in axes:
        ax.set_facecolor("#1a1d2e")

    sent_counts = df["sentiment_label"].value_counts()
    colors = [PALETTE.get(l, "#888") for l in sent_counts.index]
    bars = axes[0].bar(sent_counts.index, sent_counts.values, color=colors, edgecolor="white", linewidth=0.4)
    for bar, val in zip(bars, sent_counts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                     f"{val:,}", ha="center", va="bottom", color="white", fontsize=10, fontweight="bold")
    axes[0].set_title("Sentiment Label Distribution", color="white", fontsize=13, fontweight="bold", pad=12)
    axes[0].set_xlabel("Sentiment", color="#aaa", fontsize=10)
    axes[0].set_ylabel("Count", color="#aaa", fontsize=10)
    axes[0].tick_params(colors="white")
    axes[0].spines[:].set_color("#333")

    topic_counts = df["topic_category"].value_counts()
    bars2 = axes[1].barh(topic_counts.index, topic_counts.values,
                          color=TOPIC_PALETTE[:len(topic_counts)], edgecolor="white", linewidth=0.4)
    for bar, val in zip(bars2, topic_counts.values):
        axes[1].text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                     f"{val:,}", va="center", color="white", fontsize=9, fontweight="bold")
    axes[1].set_title("Topic Category Distribution", color="white", fontsize=13, fontweight="bold", pad=12)
    axes[1].set_xlabel("Count", color="#aaa", fontsize=10)
    axes[1].tick_params(colors="white")
    axes[1].spines[:].set_color("#333")

    plt.tight_layout(pad=2.5)
    out = os.path.join(FIGURES_DIR, "class_distribution.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  [EDA] Saved: {out}")


def plot_text_length(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0f1117")
    for ax in axes:
        ax.set_facecolor("#1a1d2e")

    for label, grp in df.groupby("sentiment_label"):
        color = PALETTE.get(label, "#888")
        axes[0].hist(grp["word_count"].clip(upper=80), bins=40, alpha=0.65,
                     label=label, color=color, edgecolor="none")
        axes[1].hist(grp["char_count"].clip(upper=400), bins=40, alpha=0.65,
                     label=label, color=color, edgecolor="none")

    for ax, title, xlabel in zip(
        axes,
        ["Word Count Distribution by Sentiment", "Character Count Distribution by Sentiment"],
        ["Word Count (clipped at 80)", "Character Count (clipped at 400)"],
    ):
        ax.set_title(title, color="white", fontsize=12, fontweight="bold", pad=10)
        ax.set_xlabel(xlabel, color="#aaa", fontsize=10)
        ax.set_ylabel("Frequency", color="#aaa", fontsize=10)
        ax.tick_params(colors="white")
        ax.spines[:].set_color("#333")
        ax.legend(facecolor="#1a1d2e", edgecolor="#555", labelcolor="white", fontsize=9)

    plt.tight_layout(pad=2.5)
    out = os.path.join(FIGURES_DIR, "text_length_hist.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  [EDA] Saved: {out}")


# ──────────────────────────────────────────────
# Train/Test Split
# ──────────────────────────────────────────────

def split_data(df: pd.DataFrame) -> tuple:
    X = df["cleaned_text"].values
    y_sent = df["sentiment_label"].values
    y_topic = df["topic_category"].values
    ids = df["text_id"].values
    raw_text = df["post_text"].values

    X_train, X_test, y_train, y_test, ids_train, ids_test, raw_train, raw_test = train_test_split(
        X, y_sent, ids, raw_text,
        test_size=0.20,
        stratify=y_sent,
        random_state=RANDOM_STATE,
    )
    return X_train, X_test, y_train, y_test, ids_train, ids_test, raw_train, raw_test


# ──────────────────────────────────────────────
# Main entry point
# ──────────────────────────────────────────────

def run_preprocessing() -> dict:
    print("\n" + "=" * 60)
    print("  STAGE 1: LOADING & AUDITING DATASET")
    print("=" * 60)
    df, audit = load_and_audit(DATA_PATH)
    print(f"  Raw rows        : {audit['raw_rows']:,}")
    print(f"  Missing text    : {audit['missing_text_rows']}")
    print(f"  Empty text      : {audit['empty_text_rows']}")
    print(f"  Duplicates rmvd : {audit['duplicate_rows_removed']}")
    print(f"  Clean rows      : {audit['clean_rows']:,}")
    print(f"  Sentiment dist  : {audit['label_counts']}")
    print(f"  Topic dist      : {audit['topic_counts']}")

    print("\n" + "=" * 60)
    print("  STAGE 2: TEXT CLEANING")
    print("=" * 60)
    df, empty_post_clean = apply_cleaning(df)
    print(f"  Empty after clean: {empty_post_clean}")
    print(f"  Final rows       : {len(df):,}")
    print(f"  Avg word count   : {df['word_count'].mean():.1f}")
    print(f"  Avg char count   : {df['char_count'].mean():.1f}")

    print("\n" + "=" * 60)
    print("  STAGE 3: EDA VISUALIZATIONS")
    print("=" * 60)
    plot_class_distribution(df)
    plot_text_length(df)

    print("\n" + "=" * 60)
    print("  STAGE 4: TRAIN / TEST SPLIT (80/20 stratified, seed=42)")
    print("=" * 60)
    X_train, X_test, y_train, y_test, ids_train, ids_test, raw_train, raw_test = split_data(df)
    print(f"  Train size : {len(X_train):,}")
    print(f"  Test size  : {len(X_test):,}")
    unique, counts = np.unique(y_train, return_counts=True)
    print("  Train label distribution:")
    for u, c in zip(unique, counts):
        print(f"    {u}: {c} ({c / len(y_train) * 100:.1f}%)")

    return {
        "df": df,
        "audit": audit,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "ids_test": ids_test,
        "raw_test": raw_test,
    }


if __name__ == "__main__":
    result = run_preprocessing()


## 2. Local Model: Training & Selection
We compare MultinomialNB, LogisticRegression, and LinearSVC using 5-fold cross-validation on the training set. The winning model is selected and tuned using GridSearchCV.


In [ ]:
"""
train.py — Round 2 NLP Model Training & Comparison
Data Vortex A'26 | Team: Event Horizon

Trains Majority-Class Baseline, MNB, Logistic Regression, and Linear SVM.
Selects winner by Macro-F1. Persists best model as .pkl.
No data leakage: TF-IDF fitted on training split only.
"""

import os
import time
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MODELS_DIR = os.path.join(os.path.dirname(__file__), "..", "models")
FIGURES_DIR = os.path.join(os.path.dirname(__file__), "..", "reports", "figures")
REPORTS_DIR = os.path.join(os.path.dirname(__file__), "..", "reports")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)


# ──────────────────────────────────────────────
# Candidate Pipelines
# ──────────────────────────────────────────────

def build_candidates() -> dict:
    tfidf_base = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50_000,
        sublinear_tf=True,
        min_df=2,
        strip_accents="unicode",
    )

    return {
        "Majority-Class Baseline": Pipeline([
            ("tfidf", TfidfVectorizer(max_features=1)),
            ("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
        ]),
        "Multinomial Naive Bayes": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2), max_features=50_000,
                sublinear_tf=False, min_df=2, strip_accents="unicode")),
            ("clf", MultinomialNB(alpha=0.5)),
        ]),
        "Logistic Regression": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2), max_features=50_000,
                sublinear_tf=True, min_df=2, strip_accents="unicode")),
            ("clf", LogisticRegression(
                C=1.0, max_iter=1000, solver="lbfgs",
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "Linear SVM": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2), max_features=50_000,
                sublinear_tf=True, min_df=2, strip_accents="unicode")),
            ("clf", LinearSVC(
                C=1.0, max_iter=2000,
                class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
    }


# ──────────────────────────────────────────────
# Model Comparison
# ──────────────────────────────────────────────

def run_model_comparison(X_train, y_train) -> tuple[pd.DataFrame, str]:
    candidates = build_candidates()
    results = []

    print("\n" + "=" * 60)
    print("  STAGE 5: MODEL CANDIDATE COMPARISON (5-Fold CV on Train)")
    print("=" * 60)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for name, pipeline in candidates.items():
        t0 = time.time()
        
        scores = cross_validate(
            pipeline, X_train, y_train, cv=skf, 
            scoring=('accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro'),
            n_jobs=-1
        )
        train_time = time.time() - t0

        acc = scores['test_accuracy'].mean()
        macro_f1 = scores['test_f1_macro'].mean()
        weighted_f1 = scores['test_f1_weighted'].mean()
        macro_prec = scores['test_precision_macro'].mean()
        macro_rec = scores['test_recall_macro'].mean()

        results.append({
            "Model": name,
            "Accuracy": round(acc, 4),
            "Macro-F1": round(macro_f1, 4),
            "Weighted-F1": round(weighted_f1, 4),
            "Macro-Precision": round(macro_prec, 4),
            "Macro-Recall": round(macro_rec, 4),
            "Train Time (s)": round(train_time, 2),
        })

        marker = " ← BASELINE" if name == "Majority-Class Baseline" else ""
        print(f"  {name:<30} Acc={acc:.4f}  MacroF1={macro_f1:.4f}  t={train_time:.2f}s{marker}")

    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values("Macro-F1", ascending=False).reset_index(drop=True)

    baseline_f1 = df_results[df_results["Model"] == "Majority-Class Baseline"]["Macro-F1"].values[0]
    baseline_acc = df_results[df_results["Model"] == "Majority-Class Baseline"]["Accuracy"].values[0]

    non_baseline = df_results[df_results["Model"] != "Majority-Class Baseline"]
    winner_name = non_baseline.iloc[0]["Model"]
    winner_f1 = non_baseline.iloc[0]["Macro-F1"]
    winner_acc = non_baseline.iloc[0]["Accuracy"]

    print(f"\n  Winner by Macro-F1: [{winner_name}]")
    print(f"  Lift over baseline: MacroF1 {winner_f1:.4f} vs {baseline_f1:.4f} "
          f"| Acc {winner_acc:.4f} vs {baseline_acc:.4f}")

    out_csv = os.path.join(REPORTS_DIR, "model_comparison.csv")
    df_results.to_csv(out_csv, index=False)
    print(f"  Comparison table saved: {out_csv}")

    return df_results, winner_name, candidates


# ──────────────────────────────────────────────
# Hyperparameter Grid Search on Winner
# ──────────────────────────────────────────────

def tune_winner(winner_name: str, X_train, y_train) -> Pipeline:
    print("\n" + "=" * 60)
    print(f"  STAGE 6: HYPERPARAMETER TUNING [{winner_name}]")
    print("=" * 60)

    if winner_name == "Logistic Regression":
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                sublinear_tf=True, strip_accents="unicode", min_df=2)),
            ("clf", LogisticRegression(
                max_iter=1000, solver="lbfgs",
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ])
        param_grid = {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [30_000, 50_000],
            "clf__C": [0.1, 1.0, 5.0, 10.0],
        }

    elif winner_name == "Linear SVM":
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                sublinear_tf=True, strip_accents="unicode", min_df=2)),
            ("clf", LinearSVC(
                max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        ])
        param_grid = {
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [30_000, 50_000],
            "clf__C": [0.1, 0.5, 1.0, 5.0],
        }

    else:
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2), max_features=50_000,
                sublinear_tf=True, min_df=2, strip_accents="unicode")),
            ("clf", MultinomialNB()),
        ])
        param_grid = {"clf__alpha": [0.1, 0.5, 1.0, 2.0]}

    gs = GridSearchCV(
        pipeline, param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=0,
        refit=True,
    )
    gs.fit(X_train, y_train)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV Macro-F1 : {gs.best_score_:.4f}")
    return gs.best_estimator_


# ──────────────────────────────────────────────
# Persist Model
# ──────────────────────────────────────────────

def save_model(model: Pipeline, name: str) -> str:
    safe_name = name.lower().replace(" ", "_").replace("-", "")
    path = os.path.join(MODELS_DIR, f"sentiment_{safe_name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"  Model saved: {path}")
    return path


# ──────────────────────────────────────────────
# Plot Model Comparison Bar Chart
# ──────────────────────────────────────────────

def plot_model_comparison(df_results: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(11, 5))
    fig.patch.set_facecolor("#0f1117")
    ax.set_facecolor("#1a1d2e")

    models = df_results["Model"].tolist()
    macro_f1 = df_results["Macro-F1"].tolist()
    accuracies = df_results["Accuracy"].tolist()

    x = np.arange(len(models))
    w = 0.35
    bars1 = ax.bar(x - w / 2, macro_f1, w, label="Macro-F1", color="#3498db", edgecolor="white", linewidth=0.4)
    bars2 = ax.bar(x + w / 2, accuracies, w, label="Accuracy", color="#2ecc71", edgecolor="white", linewidth=0.4)

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{bar.get_height():.3f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")

    ax.axhline(y=df_results[df_results["Model"] == "Majority-Class Baseline"]["Macro-F1"].values[0],
               color="#e74c3c", linestyle="--", linewidth=1.2, label="Baseline Macro-F1")

    ax.set_xticks(x)
    ax.set_xticklabels(models, color="white", fontsize=9, rotation=10, ha="right")
    ax.set_ylabel("Score", color="#aaa", fontsize=10)
    ax.set_title("Model Candidate Comparison (5-Fold CV on Train)", color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#333")
    ax.set_ylim(0, 1.0)
    ax.legend(facecolor="#1a1d2e", edgecolor="#555", labelcolor="white", fontsize=9)

    plt.tight_layout(pad=2)
    out = os.path.join(FIGURES_DIR, "model_comparison.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  [Plot] Saved: {out}")


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────

def run_training(X_train, y_train) -> dict:
    df_results, winner_name, trained_candidates = run_model_comparison(
        X_train, y_train
    )
    plot_model_comparison(df_results)

    best_model = tune_winner(winner_name, X_train, y_train)
    model_path = save_model(best_model, winner_name)

    return {
        "comparison_df": df_results,
        "winner_name": winner_name,
        "best_model": best_model,
        "model_path": model_path,
        "trained_candidates": trained_candidates,
    }


## 3. Local Model: Evaluation
We evaluate the final optimized local model on the held-out test set, generating confusion matrices, per-class metrics, and LIME explainability analysis.


In [ ]:
"""
evaluate.py — Round 2 NLP Evaluation, Confusion Matrix, Error Analysis & LIME
Data Vortex A'26 | Team: Event Horizon
"""

import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from sklearn.dummy import DummyClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
FIGURES_DIR = os.path.join(os.path.dirname(__file__), "..", "reports", "figures")
REPORTS_DIR = os.path.join(os.path.dirname(__file__), "..", "reports")
os.makedirs(FIGURES_DIR, exist_ok=True)


# ──────────────────────────────────────────────
# Full Evaluation Report
# ──────────────────────────────────────────────

def evaluate_model(model, X_test, y_test, winner_name: str, baseline_acc: float, baseline_f1: float) -> dict:
    print("\n" + "=" * 60)
    print(f"  STAGE 7: FINAL MODEL EVALUATION [{winner_name}]")
    print("=" * 60)

    y_pred = model.predict(X_test)
    labels = sorted(list(set(y_test)))

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    macro_prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    macro_rec = recall_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"  Accuracy         : {acc:.4f}  (baseline {baseline_acc:.4f}, lift +{acc-baseline_acc:.4f})")
    print(f"  Macro-F1         : {macro_f1:.4f}  (baseline {baseline_f1:.4f}, lift +{macro_f1-baseline_f1:.4f})")
    print(f"  Weighted-F1      : {weighted_f1:.4f}")
    print(f"  Macro-Precision  : {macro_prec:.4f}")
    print(f"  Macro-Recall     : {macro_rec:.4f}")
    print("\n" + classification_report(y_test, y_pred, zero_division=0))

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    return {
        "y_pred": y_pred,
        "labels": labels,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "report_dict": report,
        "baseline_acc": baseline_acc,
        "baseline_f1": baseline_f1,
    }


# ──────────────────────────────────────────────
# Confusion Matrix Plot
# ──────────────────────────────────────────────

def plot_confusion_matrix(y_test, y_pred, labels: list, winner_name: str):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0f1117")
    for ax in axes:
        ax.set_facecolor("#1a1d2e")

    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Confusion Matrix — Raw Counts", "Confusion Matrix — Normalized (Row %)"],
        ["d", ".2f"],
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, linecolor="#333",
            ax=ax, cbar=True,
            annot_kws={"size": 11, "weight": "bold", "color": "black"},
        )
        ax.set_title(title, color="white", fontsize=12, fontweight="bold", pad=10)
        ax.set_xlabel("Predicted Label", color="#aaa", fontsize=10)
        ax.set_ylabel("True Label", color="#aaa", fontsize=10)
        ax.tick_params(colors="white")

    plt.suptitle(f"Model: {winner_name}", color="#aaa", fontsize=10, y=1.02)
    plt.tight_layout(pad=2)
    out = os.path.join(FIGURES_DIR, "confusion_matrix.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  [Plot] Saved: {out}")
    return out


# ──────────────────────────────────────────────
# Error Analysis
# ──────────────────────────────────────────────

ERROR_CATEGORIES = {
    ("Negative", "Positive"): "Missed negation / sarcasm",
    ("Positive", "Negative"): "Overly negative framing detected",
    ("Neutral",  "Positive"): "Weak positive signal misread as Neutral",
    ("Neutral",  "Negative"): "Weak negative signal misread as Neutral",
    ("Positive", "Neutral"):  "Ambiguous boundary (Positive ≈ Neutral)",
    ("Negative", "Neutral"):  "Ambiguous boundary (Negative ≈ Neutral)",
}

def _categorize(true_label, pred_label, text: str) -> str:
    rule = ERROR_CATEGORIES.get((true_label, pred_label), "Other / Label noise")
    words = text.split()
    if len(words) <= 4:
        return "Extremely short text"
    if any(w in text.lower() for w in ["not", "no ", "never", "n't", "isn't", "wasn't", "won't", "can't"]):
        return "Negation handling failure"
    return rule


def run_error_analysis(X_test_raw, y_test, y_pred, n_samples: int = 20) -> pd.DataFrame:
    print("\n" + "=" * 60)
    print("  STAGE 8: ERROR ANALYSIS")
    print("=" * 60)

    mask = y_test != y_pred
    err_texts = X_test_raw[mask]
    err_true = y_test[mask]
    err_pred = y_pred[mask]

    n = min(n_samples, len(err_texts))
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(len(err_texts), size=n, replace=False)
    idx = np.sort(idx)

    rows = []
    for i in idx:
        text = str(err_texts[i])
        snippet = text[:90] + ("..." if len(text) > 90 else "")
        cat = _categorize(err_true[i], err_pred[i], text)
        rows.append({
            "Text Snippet": snippet,
            "True Label": err_true[i],
            "Predicted": err_pred[i],
            "Error Category": cat,
        })

    df_errors = pd.DataFrame(rows)

    print(f"  Total misclassified: {mask.sum()} / {len(y_test)}")
    print(f"  Sampled for analysis: {n}")
    print("\n  Error Category Breakdown:")
    for cat, cnt in df_errors["Error Category"].value_counts().items():
        print(f"    {cat:<40} {cnt}")

    out = os.path.join(REPORTS_DIR, "error_analysis.csv")
    df_errors.to_csv(out, index=False)
    print(f"  Error table saved: {out}")
    return df_errors


# ──────────────────────────────────────────────
# LIME Explanations
# ──────────────────────────────────────────────

def run_lime_explanations(model, X_test_raw, y_test, y_pred, labels: list, n=2) -> list:
    print("\n" + "=" * 60)
    print("  STAGE 9: LIME INTERPRETABILITY")
    print("=" * 60)

    try:
        from lime.lime_text import LimeTextExplainer
    except ImportError:
        print("  [WARN] lime not installed — skipping LIME step. Run: pip install lime")
        return []

    explainer = LimeTextExplainer(class_names=labels, random_state=RANDOM_STATE)

    lime_results = []

    def predict_proba_fn(texts):
        if hasattr(model, "predict_proba"):
            return model.predict_proba(texts)
        decision = model.decision_function(texts)
        exp_d = np.exp(decision - decision.max(axis=1, keepdims=True))
        return exp_d / exp_d.sum(axis=1, keepdims=True)

    correct_mask = y_test == y_pred
    error_mask = ~correct_mask

    sample_sets = [
        ("Correctly Classified", X_test_raw[correct_mask], y_test[correct_mask], y_pred[correct_mask]),
        ("Misclassified", X_test_raw[error_mask], y_test[error_mask], y_pred[error_mask]),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#0f1117")

    for ax_idx, (label_tag, texts, true_lbls, pred_lbls) in enumerate(sample_sets):
        if len(texts) == 0:
            continue
        idx = 0
        text = str(texts[idx])
        print(f"\n  [{label_tag}]")
        print(f"  Text   : {text[:100]}...")
        print(f"  True   : {true_lbls[idx]}  | Predicted: {pred_lbls[idx]}")

        exp = explainer.explain_instance(
            text, predict_proba_fn,
            num_features=10,
            num_samples=200,
            labels=[labels.index(pred_lbls[idx])],
        )

        pred_label_idx = labels.index(pred_lbls[idx])
        feature_weights = exp.as_list(label=pred_label_idx)
        words = [fw[0] for fw in feature_weights]
        weights = [fw[1] for fw in feature_weights]

        colors = ["#2ecc71" if w > 0 else "#e74c3c" for w in weights]
        ax = axes[ax_idx]
        ax.set_facecolor("#1a1d2e")
        ax.barh(words, weights, color=colors, edgecolor="white", linewidth=0.3)
        ax.axvline(0, color="white", linewidth=0.8)
        ax.set_title(f"LIME: {label_tag}\nTrue={true_lbls[idx]} | Pred={pred_lbls[idx]}",
                     color="white", fontsize=10, fontweight="bold")
        ax.tick_params(colors="white", labelsize=9)
        ax.spines[:].set_color("#333")
        ax.set_xlabel("Feature Weight", color="#aaa", fontsize=9)

        lime_results.append({
            "type": label_tag,
            "text": text[:200],
            "true_label": true_lbls[idx],
            "predicted_label": pred_lbls[idx],
            "top_features": feature_weights[:5],
        })

        print(f"  Top features: {feature_weights[:5]}")

    plt.suptitle("LIME Token-Level Feature Importance", color="white", fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout(pad=2)
    out = os.path.join(FIGURES_DIR, "lime_explanations.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"\n  [Plot] Saved: {out}")

    return lime_results


# ──────────────────────────────────────────────
# Per-Class Metrics Bar Chart
# ──────────────────────────────────────────────

def plot_per_class_metrics(report_dict: dict, labels: list):
    metrics = ["precision", "recall", "f1-score"]
    data = {m: [report_dict[l][m] for l in labels] for m in metrics}

    x = np.arange(len(labels))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#0f1117")
    ax.set_facecolor("#1a1d2e")

    colors = ["#3498db", "#2ecc71", "#e67e22"]
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        bars = ax.bar(x + i * w, data[metric], w, label=metric.capitalize(),
                      color=color, edgecolor="white", linewidth=0.4)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{bar.get_height():.2f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")

    ax.set_xticks(x + w)
    ax.set_xticklabels(labels, color="white", fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", color="#aaa", fontsize=10)
    ax.set_title("Per-Class Precision / Recall / F1", color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#333")
    ax.legend(facecolor="#1a1d2e", edgecolor="#555", labelcolor="white", fontsize=9)

    plt.tight_layout(pad=2)
    out = os.path.join(FIGURES_DIR, "per_class_metrics.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  [Plot] Saved: {out}")


# ──────────────────────────────────────────────
# Save All Metrics as JSON (for report generator)
# ──────────────────────────────────────────────

def save_metrics_json(eval_result: dict, winner_name: str, lime_results: list, df_errors: pd.DataFrame):
    out = {
        "model": winner_name,
        "accuracy": eval_result["accuracy"],
        "macro_f1": eval_result["macro_f1"],
        "weighted_f1": eval_result["weighted_f1"],
        "macro_precision": eval_result["macro_precision"],
        "macro_recall": eval_result["macro_recall"],
        "baseline_accuracy": eval_result["baseline_acc"],
        "baseline_macro_f1": eval_result["baseline_f1"],
        "lift_accuracy": round(eval_result["accuracy"] - eval_result["baseline_acc"], 4),
        "lift_macro_f1": round(eval_result["macro_f1"] - eval_result["baseline_f1"], 4),
        "per_class": eval_result["report_dict"],
        "lime_samples": lime_results,
        "error_analysis_sample": df_errors.to_dict(orient="records"),
    }
    path = os.path.join(REPORTS_DIR, "metrics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print(f"  Metrics JSON saved: {path}")
    return out


## 4. Benchmark Model: BERTweet (Colab GPU)
The following code was executed in Google Colab (T4 GPU) to train a Transformer benchmark (`vinai/bertweet-base`). It trains the model for 5 epochs and outputs the final predictions and metrics.


In [ ]:
"""
bertweet_colab.py — Self-contained Colab Script
Data Vortex A'26 | Team: Event Horizon

Run this as a single cell in Google Colab.
No local imports required.
Steps:
  1. Upload Labeled_Social_NLP_Training_Data.csv
  2. Dedup + clean 70/10/20 split
  3. Fine-tune BERTweet
  4. Evaluate on held-out test set
  5. Download predictions CSV + benchmark JSON
"""

import os
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
from datasets import Dataset
from google.colab import files as colab_files

RANDOM_STATE = 42
LABEL_MAP = {"Negative": 0, "Neutral": 1, "Positive": 2}
LABEL_NAMES = ["Negative", "Neutral", "Positive"]
MODEL_NAME = "vinai/bertweet-base"
MAX_LENGTH = 128

print("Upload Labeled_Social_NLP_Training_Data.csv")
uploaded = colab_files.upload()
data_path = list(uploaded.keys())[0]

df = pd.read_csv(data_path, encoding="utf-8")
df = df.dropna(subset=["post_text"])
df = df[df["post_text"].str.strip() != ""]
before = len(df)
df = df.drop_duplicates(subset=["post_text"])
print(f"Rows after dedup: {len(df):,}  (removed {before - len(df)} duplicates)")

df["label"] = df["sentiment_label"].map(LABEL_MAP)
df = df.dropna(subset=["label"])

X = df["post_text"].values
y = df["label"].values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.125, stratify=y_trainval, random_state=RANDOM_STATE
)

train_overlap = set(X_train).intersection(set(X_test))
val_overlap   = set(X_val).intersection(set(X_test))
assert len(train_overlap) == 0, f"LEAKAGE: {len(train_overlap)} train/test overlaps!"
assert len(val_overlap)   == 0, f"LEAKAGE: {len(val_overlap)} val/test overlaps!"
print(f"AUDIT PASSED: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)} | Zero overlap")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)

def tokenize(texts, labels):
    ds = Dataset.from_dict({"text": list(texts), "label": list(labels)})
    return ds.map(
        lambda b: tokenizer(b["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH),
        batched=True,
    )

train_tok = tokenize(X_train, y_train)
val_tok   = tokenize(X_val,   y_val)
test_tok  = tokenize(X_test,  y_test)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

args = TrainingArguments(
    output_dir="./results_bertweet_clean",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

test_out = trainer.predict(test_tok)
logits   = test_out.predictions
y_pred   = np.argmax(logits, axis=-1)
probs    = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()

macro_f1 = f1_score(y_test, y_pred, average="macro")
acc      = accuracy_score(y_test, y_pred)
kappa    = cohen_kappa_score(y_test, y_pred)

print(f"\nMacro-F1 : {macro_f1:.4f}")
print(f"Accuracy  : {acc:.4f}")
print(f"Kappa     : {kappa:.4f}")
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

preds_df = pd.DataFrame({
    "text":         X_test,
    "y_true":       y_test,
    "y_pred":       y_pred,
    "prob_Negative": probs[:, 0],
    "prob_Neutral":  probs[:, 1],
    "prob_Positive": probs[:, 2],
})
preds_df.to_csv("bertweet_predictions.csv", index=False)

summary = {
    "model": MODEL_NAME,
    "split": "70_10_20_clean_dedup",
    "macro_f1": round(float(macro_f1), 6),
    "accuracy": round(float(acc), 6),
    "kappa": round(float(kappa), 6),
    "test_size": int(len(X_test)),
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
}
with open("bertweet_benchmark.json", "w") as f:
    json.dump(summary, f, indent=2)

colab_files.download("bertweet_predictions.csv")
colab_files.download("bertweet_benchmark.json")
print("\nDONE. Save both files to round2/reports/ on your local machine.")
